# 🏆 TSTR/TRTS Model-Based Validation Framework

**The Gold Standard for Validating Augmented Data Quality**

This notebook implements the scientifically rigorous TSTR/TRTS framework to validate augmented/synthetic data quality.

---

## 📚 What is TSTR/TRTS?

| Method | Concept | What it Proves |
|--------|---------|----------------|
| **TSTR** (Train on Synthetic, Test on Real) | Train a model only on augmented/synthetic data, then test on real, held-out data | **Usability**: If the model learns generalized patterns from augmentation that apply to the real world, your data is high quality |
| **TRTS** (Train on Real, Test on Synthetic) | Train a model on real data and test it on your augmented data | **Realism**: If the model fails on augmented data, your augmentation has drifted too far from the real distribution (manifold) |

---

## 🎯 Gap Analysis

- **Small TSTR-TRTS gap** → High quality augmentation
- **Large gap** → Distribution mismatch between real and synthetic data

---

**Let's get started!**


## 🔧 Setup & Installation


In [ ]:
# Install required packages (run this in Colab)
# !pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn underthesea


In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally")


In [ ]:
import os
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# Underthesea for Vietnamese word segmentation
from underthesea import word_tokenize

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"✓ Underthesea word tokenizer loaded")


## ⚙️ Configuration


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these paths and parameters
# ============================================================================

# Data paths
if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/thesis/data/data_processed/'
    OUTPUT_DIR = '/content/drive/MyDrive/thesis/tstr_trts_results'
else:
    DATA_DIR = '../data/data_processed'
    OUTPUT_DIR = './tstr_trts_results'

# Data files for dual-track comparison
ORIGINAL_TRAIN_PATH = os.path.join(DATA_DIR, 'train_processed.csv')
COMBINED_TRAIN_PATH = os.path.join(DATA_DIR, 'train_1071_para_final.csv')
ORIGINAL_TEST_PATH = os.path.join(DATA_DIR, 'test_processed.csv')
GENERATED_TEST_PATH = os.path.join(DATA_DIR, 'test_gen_final_filtered.csv')

# Model configuration
MODEL_NAME = 'uitnlp/CafeBERT'  # CafeBERT - Vietnamese BERT model
MAX_LENGTH = 256

# Tokenization configuration
USE_UNDERTHESEA = False  # Use default tokenizer for CafeBERT (no word segmentation needed)

# Training configuration
NUM_EPOCHS = 3
BATCH_SIZE = 128
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
RANDOM_SEED = 42

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📁 Configuration:")
print(f"   Original train data: {ORIGINAL_TRAIN_PATH}")
print(f"   Combined train data: {COMBINED_TRAIN_PATH}")
print(f"   Original test data: {ORIGINAL_TEST_PATH}")
print(f"   Generated test data: {GENERATED_TEST_PATH}")
print(f"   Model: {MODEL_NAME}")
print(f"   Use Underthesea: {USE_UNDERTHESEA}")
print(f"   Output: {OUTPUT_DIR}")


In [ ]:
# ============================================================================
# TEST TOKENIZER & MODEL
# ============================================================================
# Test CafeBERT tokenizer (no word segmentation needed)

from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize as underthesea_word_tokenize
import torch

# Load model and tokenizer
model = AutoModel.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test text
test_text = 'Cà phê được trồng nhiều ở khu vực Tây Nguyên của Việt Nam.'

# ============================================================================
# TOKENIZER OPTION: Switch between 'default' and 'underthesea'
# CafeBERT works best with default tokenizer (no word segmentation)
# ============================================================================
TOKENIZER_MODE = 'default'  # Options: 'default' (recommended for CafeBERT) or 'underthesea'

def tokenize_text(text: str, mode: str = 'underthesea') -> str:
    """
    Tokenize text based on selected mode.
    
    Args:
        text: Input Vietnamese text
        mode: 'default' - use PhoBERT tokenizer directly
              'underthesea' - apply Underthesea word segmentation first (recommended for PhoBERT)
    
    Returns:
        Processed text ready for BERT tokenization
    """
    if mode == 'underthesea':
        # Apply Underthesea word segmentation (e.g., "học sinh" -> "học_sinh")
        return underthesea_word_tokenize(text, format="text")
    else:
        # Use text as-is for PhoBERT tokenizer
        return text

# Test both tokenization modes
print("🧪 Testing PhoBERT-large Tokenizer Modes...")
print("=" * 60)

for mode in ['default', 'underthesea']:
    print(f"\n📌 Mode: {mode.upper()}")
    print("-" * 40)
    
    processed_text = tokenize_text(test_text, mode=mode)
    print(f"Input text:     {test_text}")
    print(f"Processed text: {processed_text}")
    
    encoding = tokenizer(processed_text, return_tensors='pt')
    print(f"Token IDs:      {encoding['input_ids']}")
    print(f"Num tokens:     {encoding['input_ids'].shape[1]}")

# Test model inference with selected mode
print(f"\n{'=' * 60}")
print(f"🔧 Using TOKENIZER_MODE = '{TOKENIZER_MODE}' for experiments")
print("=" * 60)

processed_text = tokenize_text(test_text, mode=TOKENIZER_MODE)
encoding = tokenizer(processed_text, return_tensors='pt')

with torch.no_grad():
    output = model(**encoding)

print(f"\n📊 Model Output:")
print(f"   Last hidden state shape: {output.last_hidden_state.shape}")
print(f"   Pooler output shape: {output.pooler_output.shape}")
print(f"✅ {MODEL_NAME} tokenizer and model test passed!")

## 📊 Data Classes & Utilities


In [ ]:
@dataclass
class ValidationResult:
    """Container for validation experiment results"""
    experiment_name: str
    train_source: str  # 'real', 'synthetic', or 'mixed'
    test_source: str   # 'real' or 'synthetic'
    accuracy: float
    precision_weighted: float
    recall_weighted: float
    f1_weighted: float
    f1_macro: float
    per_class_metrics: Dict[str, Dict[str, float]]
    confusion_matrix: np.ndarray
    training_time: float
    num_train_samples: int
    num_test_samples: int


class EmotionDataset(Dataset):
    """PyTorch Dataset for emotion classification with optional Underthesea tokenization"""
    
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128, use_underthesea: bool = False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_underthesea = use_underthesea
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Apply Underthesea word segmentation if enabled
        if self.use_underthesea:
            text = tokenize_text(text, mode='underthesea')
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


# Test tokenization modes
print("🔤 Tokenization example:")
sample_text = "Tôi rất vui khi được học tiếng Việt"
print(f"   Original:    {sample_text}")
print(f"   Default:     {tokenize_text(sample_text, mode='default')}")
print(f"   Underthesea: {tokenize_text(sample_text, mode='underthesea')}")


## 🔬 TSTR/TRTS Validator Class


In [ ]:
class TSTRTRTSValidator:
    """
    Validation framework for comparing model performance across
    multiple train/test dataset combinations.
    """

    def __init__(
        self,
        model_name: str = 'vinai/phobert-large',
        output_dir: str = './tstr_trts_results',
        max_length: int = 128,
        random_seed: int = 42,
        use_underthesea: bool = True
    ):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        self.random_seed = random_seed
        self.use_underthesea = use_underthesea
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        print("🔧 Validator initialized")
        print(f"   Model: {model_name}")
        print(f"   Device: {self.device}")
        print(f"   Use Underthesea: {use_underthesea}")

        os.makedirs(output_dir, exist_ok=True)

        self.tokenizer = None
        self.label2id = None
        self.id2label = None
        self.results: List[ValidationResult] = []

    def _setup_label_mappings(self, labels: pd.Series):
        """Setup label to ID mappings from all datasets"""
        unique_labels = sorted(labels.unique())
        self.label2id = {label: idx for idx, label in enumerate(unique_labels)}
        self.id2label = {idx: label for label, idx in self.label2id.items()}
        print(f"   Labels: {list(self.label2id.keys())}")

    def _standardize_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        """Standardize dataset columns to text/label schema"""
        vn_to_en = {
            'vui vẻ': 'Enjoyment', 'khó chịu': 'Disgust', 'khác': 'Other',
            'buồn': 'Sadness', 'giận dữ': 'Anger', 'sợ hãi': 'Fear',
            'ngạc nhiên': 'Surprise', 'buồn bã': 'Sadness'
        }

        text_col = None
        for col in ['Sentence_clean', 'Sentence', 'text']:
            if col in df.columns:
                text_col = col
                break

        label_col = None
        for col in ['Emotion', 'emotion_vn', 'label']:
            if col in df.columns:
                label_col = col
                break

        if text_col is None or label_col is None:
            raise ValueError(
                f"Cannot detect required columns. Available columns: {df.columns.tolist()}"
            )

        out_df = df.rename(columns={text_col: 'text', label_col: 'label'})[['text', 'label']].copy()
        out_df['label'] = out_df['label'].apply(lambda x: vn_to_en.get(x, x))
        out_df['text'] = out_df['text'].astype(str)

        # Remove rows with missing values to avoid tokenizer/training failures.
        out_df = out_df.dropna(subset=['text', 'label']).reset_index(drop=True)
        return out_df

    def load_datasets(self, dataset_paths: Dict[str, str]) -> Dict[str, pd.DataFrame]:
        """Load and standardize multiple datasets, then build unified label mapping"""
        print("\n📊 Loading datasets...")
        datasets = {}

        for name, path in dataset_paths.items():
            df = pd.read_csv(path)
            df = self._standardize_dataframe(df)
            datasets[name] = df
            print(f"   {name}: {len(df)} samples")

        all_labels = pd.concat([df['label'] for df in datasets.values()], ignore_index=True)
        self._setup_label_mappings(all_labels)

        return datasets

    def _create_dataset(self, df: pd.DataFrame, text_col: str = 'text', label_col: str = 'label') -> EmotionDataset:
        """Create PyTorch dataset from DataFrame with optional Underthesea tokenization"""
        texts = df[text_col].tolist()
        labels = [self.label2id[label] for label in df[label_col]]
        return EmotionDataset(texts, labels, self.tokenizer, self.max_length, use_underthesea=self.use_underthesea)

    def _compute_metrics(self, eval_pred) -> Dict[str, float]:
        """Compute metrics for Trainer"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted', zero_division=0
        )
        f1_macro = precision_recall_fscore_support(
            labels, predictions, average='macro', zero_division=0
        )[2]
        accuracy = accuracy_score(labels, predictions)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'f1_macro': f1_macro
        }

    def train_and_evaluate(
        self,
        train_df: pd.DataFrame,
        test_df: pd.DataFrame,
        experiment_name: str,
        train_source: str,
        test_source: str,
        num_epochs: int = 3,
        batch_size: int = 16,
        learning_rate: float = 2e-5,
        warmup_ratio: float = 0.1
    ) -> ValidationResult:
        """Train a model and evaluate on test set"""
        print(f"\n{'='*60}")
        print(f"🔬 Experiment: {experiment_name}")
        print(f"   Train: {train_source} ({len(train_df)} samples)")
        print(f"   Test: {test_source} ({len(test_df)} samples)")
        print(f"{'='*60}")

        start_time = datetime.now()

        if self.tokenizer is None:
            print("   Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)

        train_dataset = self._create_dataset(train_df)
        test_dataset = self._create_dataset(test_df)

        train_size = int(0.9 * len(train_dataset))
        val_size = len(train_dataset) - train_size
        train_subset, val_subset = torch.utils.data.random_split(
            train_dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(self.random_seed)
        )

        print("   Loading model...")
        model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=len(self.label2id),
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True
        )
        model = model.to(self.device)

        training_args = TrainingArguments(
            output_dir=os.path.join(self.output_dir, experiment_name),
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            warmup_ratio=warmup_ratio,
            learning_rate=learning_rate,
            weight_decay=0.01,
            logging_dir=os.path.join(self.output_dir, experiment_name, 'logs'),
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='no',
            load_best_model_at_end=False,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            seed=self.random_seed,
            report_to='none',
            fp16=torch.cuda.is_available(),
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_subset,
            eval_dataset=val_subset,
            compute_metrics=self._compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        print("   Training...")
        trainer.train()

        print("   Evaluating on test set...")
        predictions = trainer.predict(test_dataset)
        preds = np.argmax(predictions.predictions, axis=1)
        labels = [self.label2id[label] for label in test_df['label']]

        accuracy = accuracy_score(labels, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average='weighted', zero_division=0
        )
        f1_macro = precision_recall_fscore_support(
            labels, preds, average='macro', zero_division=0
        )[2]

        report = classification_report(
            labels, preds,
            target_names=list(self.id2label.values()),
            output_dict=True,
            zero_division=0
        )

        per_class = {}
        for label_name in self.id2label.values():
            if label_name in report:
                per_class[label_name] = {
                    'precision': report[label_name]['precision'],
                    'recall': report[label_name]['recall'],
                    'f1-score': report[label_name]['f1-score'],
                    'support': report[label_name]['support']
                }

        cm = confusion_matrix(labels, preds)
        training_time = (datetime.now() - start_time).total_seconds()

        result = ValidationResult(
            experiment_name=experiment_name,
            train_source=train_source,
            test_source=test_source,
            accuracy=accuracy,
            precision_weighted=precision,
            recall_weighted=recall,
            f1_weighted=f1,
            f1_macro=f1_macro,
            per_class_metrics=per_class,
            confusion_matrix=cm,
            training_time=training_time,
            num_train_samples=len(train_df),
            num_test_samples=len(test_df)
        )

        self.results.append(result)

        print("\n   📈 Results:")
        print(f"      Accuracy: {accuracy:.4f}")
        print(f"      F1 (weighted): {f1:.4f}")
        print(f"      F1 (macro): {f1_macro:.4f}")
        print(f"      Training time: {training_time:.1f}s")

        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        return result


## 📈 Analysis & Reporting Functions


In [ ]:
def analyze_results(results: List[ValidationResult], label2id: Dict) -> Dict[str, Any]:
    """Analyze 4-run dual-track results and compute averaged comparison metrics"""
    if len(results) < 4:
        print("⚠️ Expected 4 experiments for dual-track analysis")
        return {}

    analysis = {
        'summary': {},
        'train_track_averages': {},
        'test_track_averages': {},
        'delta_combined_minus_original': {}
    }

    for r in results:
        analysis['summary'][r.experiment_name] = {
            'train_source': r.train_source,
            'test_source': r.test_source,
            'accuracy': r.accuracy,
            'f1_weighted': r.f1_weighted,
            'f1_macro': r.f1_macro,
            'train_samples': r.num_train_samples,
            'test_samples': r.num_test_samples,
        }

    metric_cols = ['accuracy', 'f1_weighted', 'f1_macro']
    summary_df = pd.DataFrame.from_dict(analysis['summary'], orient='index').reset_index()
    summary_df = summary_df.rename(columns={'index': 'experiment_name'})

    # Average across test sets for each training track
    train_avg_df = summary_df.groupby('train_source', as_index=False)[metric_cols].mean()
    for _, row in train_avg_df.iterrows():
        analysis['train_track_averages'][row['train_source']] = {
            'accuracy': row['accuracy'],
            'f1_weighted': row['f1_weighted'],
            'f1_macro': row['f1_macro'],
        }

    # Average across train sets for each test track
    test_avg_df = summary_df.groupby('test_source', as_index=False)[metric_cols].mean()
    for _, row in test_avg_df.iterrows():
        analysis['test_track_averages'][row['test_source']] = {
            'accuracy': row['accuracy'],
            'f1_weighted': row['f1_weighted'],
            'f1_macro': row['f1_macro'],
        }

    # Delta: CombinedTrain average - OriginalTrain average
    orig = analysis['train_track_averages'].get('OriginalTrain')
    comb = analysis['train_track_averages'].get('CombinedTrain')
    if orig and comb:
        analysis['delta_combined_minus_original'] = {
            'accuracy': comb['accuracy'] - orig['accuracy'],
            'f1_weighted': comb['f1_weighted'] - orig['f1_weighted'],
            'f1_macro': comb['f1_macro'] - orig['f1_macro'],
        }

    return analysis


def generate_report(results: List[ValidationResult], analysis: Dict, model_name: str, output_dir: str) -> str:
    """Generate a comprehensive dual-track validation report"""
    report_lines = [
        "=" * 80,
        "DUAL-TRACK VALIDATION REPORT",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Model: {model_name}",
        "=" * 80,
        "",
        "📊 RAW EXPERIMENT RESULTS (4 RUNS)",
        "-" * 80,
        f"{'Experiment':<16} {'Train':<15} {'Test':<15} {'Accuracy':>10} {'F1-W':>10} {'F1-M':>10}",
        "-" * 80,
    ]

    for r in results:
        report_lines.append(
            f"{r.experiment_name:<16} {r.train_source:<15} {r.test_source:<15} "
            f"{r.accuracy:>10.4f} {r.f1_weighted:>10.4f} {r.f1_macro:>10.4f}"
        )

    report_lines.extend(["", "📈 AVERAGE BY TRAINING TRACK", "-" * 80])
    for track, metrics in analysis.get('train_track_averages', {}).items():
        report_lines.append(
            f"{track:<15} Accuracy={metrics['accuracy']:.4f} | "
            f"F1-W={metrics['f1_weighted']:.4f} | F1-M={metrics['f1_macro']:.4f}"
        )

    report_lines.extend(["", "📌 DELTA (CombinedTrainAvg - OriginalTrainAvg)", "-" * 80])
    delta = analysis.get('delta_combined_minus_original', {})
    if delta:
        report_lines.append(f"Accuracy Δ: {delta['accuracy']:+.4f}")
        report_lines.append(f"F1 Weighted Δ: {delta['f1_weighted']:+.4f}")
        report_lines.append(f"F1 Macro Δ: {delta['f1_macro']:+.4f}")
    else:
        report_lines.append("Delta unavailable (missing train-track averages)")

    report_lines.extend(["", "📎 NOTE", "-" * 80])
    report_lines.append(
        "Positive delta means combined synthetic+real training improves over original-only training on average across both test sets."
    )
    report_lines.extend(["", "=" * 80])

    report = "\n".join(report_lines)
    print(report)

    with open(os.path.join(output_dir, 'validation_report.txt'), 'w') as f:
        f.write(report)

    json_results = {
        'experiments': [
            {
                'name': r.experiment_name,
                'train_source': r.train_source,
                'test_source': r.test_source,
                'accuracy': r.accuracy,
                'f1_weighted': r.f1_weighted,
                'f1_macro': r.f1_macro,
                'train_samples': r.num_train_samples,
                'test_samples': r.num_test_samples,
            }
            for r in results
        ],
        'analysis': analysis,
    }
    with open(os.path.join(output_dir, 'validation_results.json'), 'w') as f:
        json.dump(json_results, f, indent=2)

    print(f"\n💾 Report saved to: {output_dir}")
    return report


In [ ]:
def plot_results(results: List[ValidationResult], id2label: Dict, output_dir: str):
    """Generate visualization plots for dual-track validation results"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    summary_df = pd.DataFrame([
        {
            'experiment_name': r.experiment_name,
            'train_source': r.train_source,
            'test_source': r.test_source,
            'accuracy': r.accuracy,
            'f1_weighted': r.f1_weighted,
            'f1_macro': r.f1_macro,
        }
        for r in results
    ])

    # 1) Macro F1 across 4 raw experiments
    ax1 = axes[0, 0]
    exp_names = summary_df['experiment_name'].tolist()
    f1_scores = summary_df['f1_macro'].tolist()
    bars = ax1.bar(exp_names, f1_scores, color='#3498db')
    ax1.set_ylabel('F1 Score (Macro)')
    ax1.set_title('Raw Experiment Results (Macro F1)')
    ax1.set_ylim(0, 1)
    ax1.tick_params(axis='x', rotation=20)
    for bar, score in zip(bars, f1_scores):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f'{score:.3f}',
            ha='center',
            va='bottom',
            fontsize=9,
        )

    # 2) Averaged metrics per training track
    ax2 = axes[0, 1]
    train_avg = summary_df.groupby('train_source', as_index=False)[['accuracy', 'f1_weighted', 'f1_macro']].mean()
    x = np.arange(len(train_avg))
    width = 0.25
    ax2.bar(x - width, train_avg['accuracy'], width, label='Accuracy', color='#3498db')
    ax2.bar(x, train_avg['f1_weighted'], width, label='F1 (Weighted)', color='#2ecc71')
    ax2.bar(x + width, train_avg['f1_macro'], width, label='F1 (Macro)', color='#e74c3c')
    ax2.set_xticks(x)
    ax2.set_xticklabels(train_avg['train_source'].tolist(), rotation=15)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel('Average Score')
    ax2.set_title('Averages by Training Track')
    ax2.legend()

    # 3) Confusion matrix for ORIG_to_ORIG
    ax3 = axes[1, 0]
    orig_to_orig = next((r for r in results if r.train_source == 'OriginalTrain' and r.test_source == 'OriginalTest'), None)
    if orig_to_orig is not None:
        sns.heatmap(
            orig_to_orig.confusion_matrix,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=list(id2label.values()),
            yticklabels=list(id2label.values()),
            ax=ax3,
        )
        ax3.set_title('Confusion Matrix: ORIG_to_ORIG')
        ax3.set_xlabel('Predicted')
        ax3.set_ylabel('Actual')

    # 4) Confusion matrix for COMB_to_GEN
    ax4 = axes[1, 1]
    comb_to_gen = next((r for r in results if r.train_source == 'CombinedTrain' and r.test_source == 'GeneratedTest'), None)
    if comb_to_gen is not None:
        sns.heatmap(
            comb_to_gen.confusion_matrix,
            annot=True,
            fmt='d',
            cmap='Purples',
            xticklabels=list(id2label.values()),
            yticklabels=list(id2label.values()),
            ax=ax4,
        )
        ax4.set_title('Confusion Matrix: COMB_to_GEN')
        ax4.set_xlabel('Predicted')
        ax4.set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'validation_plots.png'), dpi=150, bbox_inches='tight')
    print(f"📊 Plot saved to: {output_dir}/validation_plots.png")
    plt.show()


## 🚀 Run Validation Suite

This section runs 4 experiments for dual-track comparison:
1. **ORIG_to_ORIG**: Train on original → Test on original test set
2. **ORIG_to_GEN**: Train on original → Test on generated test set
3. **COMB_to_ORIG**: Train on combined (synthetic + real) → Test on original test set
4. **COMB_to_GEN**: Train on combined (synthetic + real) → Test on generated test set


In [ ]:
# Initialize validator
validator = TSTRTRTSValidator(
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
    max_length=MAX_LENGTH,
    random_seed=RANDOM_SEED
)


In [ ]:
# Load all datasets used in dual-track comparison
dataset_paths = {
    'original_train': ORIGINAL_TRAIN_PATH,
    'combined_train': COMBINED_TRAIN_PATH,
    'original_test': ORIGINAL_TEST_PATH,
    'generated_test': GENERATED_TEST_PATH,
}

datasets = validator.load_datasets(dataset_paths)

original_train_df = datasets['original_train']
combined_train_df = datasets['combined_train']
original_test_df = datasets['original_test']
generated_test_df = datasets['generated_test']

# Display data distributions
print("\n📊 Data Distributions:")
for name, df in datasets.items():
    print(f"\n{name} ({len(df)} samples):")
    print(df['label'].value_counts())


In [ ]:
# Define train and test tracks for experiment matrix
train_sets = {
    'OriginalTrain': original_train_df,
    'CombinedTrain': combined_train_df,
}

test_sets = {
    'OriginalTest': original_test_df,
    'GeneratedTest': generated_test_df,
}

print("✅ Experiment matrix prepared:")
print(f"   Train tracks: {list(train_sets.keys())}")
print(f"   Test tracks: {list(test_sets.keys())}")


### Run 4 Experiments (2 Train Tracks × 2 Test Tracks)

Each run trains a fresh model, then evaluates on the selected test set.


In [ ]:
experiment_results = []

train_code = {'OriginalTrain': 'ORIG', 'CombinedTrain': 'COMB'}
test_code = {'OriginalTest': 'ORIG', 'GeneratedTest': 'GEN'}

for train_name, train_df in train_sets.items():
    for test_name, test_df in test_sets.items():
        experiment_name = f"{train_code[train_name]}_to_{test_code[test_name]}"
        result = validator.train_and_evaluate(
            train_df=train_df,
            test_df=test_df,
            experiment_name=experiment_name,
            train_source=train_name,
            test_source=test_name,
            num_epochs=NUM_EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            warmup_ratio=WARMUP_RATIO,
        )
        experiment_results.append(result)

print(f"\n✅ Completed {len(experiment_results)} experiments")


### Experiment Runs Completed

Proceed to aggregate metrics for side-by-side comparison between training tracks.

In [ ]:
# Results are already stored in validator.results and experiment_results
# Keep a quick check table for the 4 raw runs
raw_check_df = pd.DataFrame([
    {
        'Experiment': r.experiment_name,
        'Train Source': r.train_source,
        'Test Source': r.test_source,
        'Accuracy': round(r.accuracy, 4),
        'F1 (Weighted)': round(r.f1_weighted, 4),
        'F1 (Macro)': round(r.f1_macro, 4),
    }
    for r in experiment_results
])
raw_check_df


## 📊 Results Analysis & Report


In [ ]:
# Analyze 4-run dual-track results
analysis = analyze_results(validator.results, validator.label2id)

# Generate report with raw results, train-track averages, and delta
report = generate_report(
    results=validator.results,
    analysis=analysis,
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
)


## 📊 Visualization


In [ ]:
# Generate visualization plots
plot_results(
    results=validator.results,
    id2label=validator.id2label,
    output_dir=OUTPUT_DIR
)


In [ ]:
# Create summary DataFrame with 4 raw rows + 2 averaged rows
summary_data = []
for r in validator.results:
    summary_data.append({
        'Experiment': r.experiment_name,
        'Train Source': r.train_source,
        'Test Source': r.test_source,
        'Accuracy': round(r.accuracy, 4),
        'F1 (Weighted)': round(r.f1_weighted, 4),
        'F1 (Macro)': round(r.f1_macro, 4),
        'Train Samples': r.num_train_samples,
        'Test Samples': r.num_test_samples,
        'Time (s)': round(r.training_time, 1)
    })

summary_df = pd.DataFrame(summary_data)

avg_rows = []
for train_source, metrics in analysis.get('train_track_averages', {}).items():
    avg_rows.append({
        'Experiment': f"AVG_{train_source}",
        'Train Source': train_source,
        'Test Source': 'Average(OriginalTest,GeneratedTest)',
        'Accuracy': round(metrics['accuracy'], 4),
        'F1 (Weighted)': round(metrics['f1_weighted'], 4),
        'F1 (Macro)': round(metrics['f1_macro'], 4),
        'Train Samples': '-',
        'Test Samples': '-',
        'Time (s)': '-'
    })

summary_with_avg_df = pd.concat([summary_df, pd.DataFrame(avg_rows)], ignore_index=True)
summary_with_avg_df


## 🎯 Dual-Track Comparison Summary


In [ ]:
print("\n" + "=" * 80)
print("🎯 DUAL-TRACK AVERAGE COMPARISON")
print("=" * 80)

train_avgs = analysis.get('train_track_averages', {})
for track, metrics in train_avgs.items():
    print(f"\n📌 {track}")
    print(f"   Avg Accuracy:      {metrics['accuracy']:.4f}")
    print(f"   Avg F1 (Weighted): {metrics['f1_weighted']:.4f}")
    print(f"   Avg F1 (Macro):    {metrics['f1_macro']:.4f}")

delta = analysis.get('delta_combined_minus_original', {})
if delta:
    print("\n📌 DELTA (CombinedTrainAvg - OriginalTrainAvg)")
    print(f"   Accuracy Δ:      {delta['accuracy']:+.4f}")
    print(f"   F1 (Weighted) Δ: {delta['f1_weighted']:+.4f}")
    print(f"   F1 (Macro) Δ:    {delta['f1_macro']:+.4f}")

    if delta['f1_macro'] > 0:
        print("   ✅ Combined training improves macro F1 on average")
    elif delta['f1_macro'] < 0:
        print("   ⚠️ Combined training reduces macro F1 on average")
    else:
        print("   ➖ No macro F1 change on average")

print("\n" + "=" * 80)


## 📝 Interpretation Guide

### Per-Track Averages
- Compare `OriginalTrain` average vs `CombinedTrain` average over both test sets.
- Higher average values indicate better overall generalization across original and generated test distributions.

### Delta Interpretation (`CombinedTrainAvg - OriginalTrainAvg`)
- **Positive delta**: combined synthetic+real training improves performance.
- **Near zero delta**: little to no measurable change.
- **Negative delta**: combined training hurts average performance.

### Metric Priority
- Use **F1 (Macro)** as the main metric for balanced class-level performance.
- Use **F1 (Weighted)** and **Accuracy** as supporting metrics.


In [ ]:
# End of validation notebook
print("✅ TSTR/TRTS Validation Complete!")


# Keep Google Colab session alive
import time
from google.colab import runtime

def auto_disconnect():
    """Automatically disconnect Google Colab runtime after a delay"""
    time.sleep(10)  # Wait 5 minutes after completion
    print("🔌 Auto-disconnecting Colab runtime...")
    runtime.unassign()

# Start the auto-disconnect thread
if 'google.colab' in str(get_ipython()):
    import threading
    disconnect_thread = threading.Thread(target=auto_disconnect, daemon=True)
    disconnect_thread.start()
    print("⏰ Auto-disconnect scheduled (runtime will disconnect in 5 minutes)")

